# E-commerce: Data Exploration with DuckDB

In this notebook we load 5 CSV files into tables using DuckDB 
and explore them with SQL designing the data warehouse.

**What we will do**

1. Load the CSV files into DuckDB
2. Understand each table: structure, grain, keys
3. Check relationships (foreign keys)
4. Find the data quality issues
5. Preview the star schema
6. Run OLAP-style queries (roll-up, slice, dice, drill-down)

**Tables**: 'customers', 'orders', 'order_items', 'products', 'sellers'

In [ ]:
from pathlib import Path

import pandas as pd
import duckdb

In [ ]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
WAREHOUSE_DIR = DATA_DIR / "warehouse"

csv = RAW_DIR / "olist_orders_dataset.csv"

In [ ]:
print(duckdb.sql(f"SELECT count(*) from '{csv}'"))

## DuckDB lets us query CSV data with SQL. The database file persists locally so later notebooks can reuse loaded raw tables.

In [ ]:
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

## Define helper function

In [ ]:
def q(sql:str) -> pd.DataFrame:
    """ Runs a SQL query in DuckDB and returns a DataFrame."""
    return con.sql(sql).df()

## 2. Load the source tables into a **raw schema**

A schema is a namespace that groups tables. We keep the source copies in **raw** so that we can differentiate it from the cleaned warehouse tables.

In [ ]:
RAW_TABLES: dict[str, dict[str, str]] = {
    "customers": {"customer_zip_code_prefix": "VARCHAR"}, # keep leading zeros
    "orders": {},
    "order_items": {},
    "products": {},
    "sellers": {}
}

In [ ]:
# Define raw schema
RAW_SCHEMA = "raw"

In [ ]:
def csv_path(table:str) -> Path:
    return RAW_DIR /f"olist_{table}_dataset.csv"

In [ ]:
# Load raw data into tables
def load_raw_table(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    """Create a raw.<table> from its csv and return the raw count"""
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"DROP TABLE IF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

In [ ]:
# Create the schema
con.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")

In [ ]:
row_counts = {t: load_raw_table(con, t, types) for t, types in RAW_TABLES.items()}

In [ ]:
pd.DataFrame(row_counts.items(), columns=["table_name", "row_count"])

In [ ]:
q("""
    SELECT 
        customer_id, 
        customer_zip_code_prefix 
    FROM raw.customers LIMIT 5
""")

In [ ]:
for t in RAW_TABLES:
    print(f"------ {t} -----")
    display(q(f"DESCRIBE raw.{t}")[["column_name", "column_type", "null"]])

---
## Exercise 1 – Explore: DESCRIBE and preview each table

Run `DESCRIBE` and `SELECT * LIMIT 5` on every table. Identify what **one row represents** (the grain).

| Table | Grain (what one row represents) |
|---|---|
| customers | One customer account registered on the platform |
| orders | One order placed by a customer |
| order_items | One line item (a product from one seller) within an order |
| products | One unique product listed on the platform |
| sellers | One seller registered on the platform |

In [ ]:
# Exercise 1 – DESCRIBE and SELECT * LIMIT 5 for every table
# Grain (what one row represents) is noted in the markdown above.
for table in ["customers", "orders", "order_items", "products", "sellers"]:
    print(f"\n{'='*60}")
    print(f"  TABLE: raw.{table}")
    print(f"{'='*60}")
    print("--- DESCRIBE ---")
    display(q(f"DESCRIBE raw.{table}"))
    print("--- SELECT * LIMIT 5 ---")
    display(q(f"SELECT * FROM raw.{table} LIMIT 5"))

---
## Exercise 2 – Primary Key Validation

For each table: `total_rows == distinct_key_values` and `nulls == 0` must both hold.  
`order_items` needs a **composite key** `(order_id, order_item_id)` — neither column alone is unique.

In [ ]:
# Exercise 2 – Validate primary keys for all five tables
# A PK is valid when: total_rows == distinct_vals  AND  nulls == 0

# --- customers: PK = customer_id ---
print("customers  PK = customer_id")
display(q("""
    SELECT
        COUNT(*)                               AS total_rows,
        COUNT(DISTINCT customer_id)            AS distinct_vals,
        COUNT(*) - COUNT(DISTINCT customer_id) AS duplicates,
        COUNT(*) - COUNT(customer_id)          AS nulls
    FROM raw.customers
"""))
-- Result: total=99441  distinct=99441  dup=0  nulls=0  -> VALID PK

# --- orders: PK = order_id ---
print("orders  PK = order_id")
display(q("""
    SELECT
        COUNT(*)                             AS total_rows,
        COUNT(DISTINCT order_id)             AS distinct_vals,
        COUNT(*) - COUNT(DISTINCT order_id)  AS duplicates,
        COUNT(*) - COUNT(order_id)           AS nulls
    FROM raw.orders
"""))
-- Result: total=99441  distinct=99441  dup=0  nulls=0  -> VALID PK

# --- order_items: COMPOSITE PK = (order_id, order_item_id) ---
print("order_items  COMPOSITE PK = (order_id, order_item_id)")
display(q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id || '|' || CAST(order_item_id AS VARCHAR)) AS distinct_pairs,
        COUNT(*) - COUNT(DISTINCT order_id || '|' || CAST(order_item_id AS VARCHAR)) AS duplicates,
        COUNT(*) - COUNT(order_id)      AS nulls_order_id,
        COUNT(*) - COUNT(order_item_id) AS nulls_item_id
    FROM raw.order_items
"""))
-- Result: total=112650  distinct_pairs=112650  dup=0  nulls=0  -> VALID composite PK

# --- products: PK = product_id ---
print("products  PK = product_id")
display(q("""
    SELECT
        COUNT(*)                              AS total_rows,
        COUNT(DISTINCT product_id)            AS distinct_vals,
        COUNT(*) - COUNT(DISTINCT product_id) AS duplicates,
        COUNT(*) - COUNT(product_id)          AS nulls
    FROM raw.products
"""))
-- Result: total=32951  distinct=32951  dup=0  nulls=0  -> VALID PK

# --- sellers: PK = seller_id ---
print("sellers  PK = seller_id")
display(q("""
    SELECT
        COUNT(*)                             AS total_rows,
        COUNT(DISTINCT seller_id)            AS distinct_vals,
        COUNT(*) - COUNT(DISTINCT seller_id) AS duplicates,
        COUNT(*) - COUNT(seller_id)          AS nulls
    FROM raw.sellers
"""))
-- Result: total=3095  distinct=3095  dup=0  nulls=0  -> VALID PK

---
## Exercise 3 – Foreign Key Integrity (Orphan Check)

An **orphan row** is a child row whose FK value has no matching PK in the parent table.  
We use `NOT EXISTS` to find orphans. Count = 0 confirms the FK relationship is valid.

| Child | FK | Parent | Parent PK |
|---|---|---|---|
| orders | customer_id | customers | customer_id |
| order_items | order_id | orders | order_id |
| order_items | product_id | products | product_id |
| order_items | seller_id | sellers | seller_id |

In [ ]:
# Exercise 3 – Orphan checks using NOT EXISTS (equivalent to LEFT ANTI JOIN)
# orphan_count = 0 means FK is satisfied; > 0 means referential integrity is broken

# orders.customer_id -> customers.customer_id
print("Orphaned orders (customer_id missing from customers):")
display(q("""
    SELECT COUNT(*) AS orphan_count
    FROM raw.orders o
    WHERE NOT EXISTS (
        SELECT 1 FROM raw.customers c
        WHERE c.customer_id = o.customer_id
    )
"""))
-- Result: 0 -> orders.customer_id is a valid FK to customers.customer_id

# order_items.order_id -> orders.order_id
print("Orphaned order_items (order_id missing from orders):")
display(q("""
    SELECT COUNT(*) AS orphan_count
    FROM raw.order_items oi
    WHERE NOT EXISTS (
        SELECT 1 FROM raw.orders o
        WHERE o.order_id = oi.order_id
    )
"""))
-- Result: 0 -> order_items.order_id is a valid FK to orders.order_id

# order_items.product_id -> products.product_id
print("Orphaned order_items (product_id missing from products):")
display(q("""
    SELECT COUNT(*) AS orphan_count
    FROM raw.order_items oi
    WHERE NOT EXISTS (
        SELECT 1 FROM raw.products p
        WHERE p.product_id = oi.product_id
    )
"""))
-- Result: 0 -> order_items.product_id is a valid FK to products.product_id

# order_items.seller_id -> sellers.seller_id
print("Orphaned order_items (seller_id missing from sellers):")
display(q("""
    SELECT COUNT(*) AS orphan_count
    FROM raw.order_items oi
    WHERE NOT EXISTS (
        SELECT 1 FROM raw.sellers s
        WHERE s.seller_id = oi.seller_id
    )
"""))
-- Result: 0 -> order_items.seller_id is a valid FK to sellers.seller_id

---
## Exercise 4 – Measure Cardinality

Count the **min and max** number of child rows per parent. This determines the crow’s foot symbol on the ERD:

| min | max | Symbol | Meaning |
|---|---|---|---|
| 0 | 1 | `o\|` | zero or one |
| 1 | 1 | `\|\|` | exactly one |
| 0 | N | `o{` | zero or many |
| 1 | N | `\|{` | one or many |

In [ ]:
# Exercise 4 – Cardinality: min/max children per parent for each FK relationship

# --- customers -> orders ---
print("customers -> orders  (orders per customer):")
display(q("""
    WITH orders_per_customer AS (
        SELECT c.customer_id,
               COUNT(o.order_id) AS order_count
        FROM raw.customers c
        LEFT JOIN raw.orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id
    )
    SELECT
        MIN(order_count) AS min_orders,
        MAX(order_count) AS max_orders,
        SUM(CASE WHEN order_count = 0 THEN 1 ELSE 0 END) AS customers_with_zero_orders
    FROM orders_per_customer
"""))
-- Result: min=1  max=1  zeros=0
-- => In this snapshot each customer_id maps to exactly one order.
-- Diagram shows customers ||--|{ orders  (logically a customer may place many orders over time)

# --- orders -> order_items ---
print("orders -> order_items  (items per order):")
display(q("""
    WITH items_per_order AS (
        SELECT o.order_id,
               COUNT(oi.order_id) AS item_count
        FROM raw.orders o
        LEFT JOIN raw.order_items oi ON o.order_id = oi.order_id
        GROUP BY o.order_id
    )
    SELECT
        MIN(item_count) AS min_items,
        MAX(item_count) AS max_items,
        SUM(CASE WHEN item_count = 0 THEN 1 ELSE 0 END) AS orders_with_zero_items
    FROM items_per_order
"""))
-- Result: min=0  max=21  zeros=775
-- => 775 orders have no items  -> orders ||--o{ order_items  (CIRCLE JUSTIFIED by data)

# --- products -> order_items ---
print("products -> order_items  (appearances per product):")
display(q("""
    WITH appearances AS (
        SELECT p.product_id,
               COUNT(oi.product_id) AS appearance_count
        FROM raw.products p
        LEFT JOIN raw.order_items oi ON p.product_id = oi.product_id
        GROUP BY p.product_id
    )
    SELECT
        MIN(appearance_count) AS min_appearances,
        MAX(appearance_count) AS max_appearances,
        SUM(CASE WHEN appearance_count = 0 THEN 1 ELSE 0 END) AS products_never_ordered
    FROM appearances
"""))
-- Result: min=1  max=527  zeros=0
-- => Every product was ordered at least once  -> products ||--|{ order_items (STRICTER than diagram)

# --- sellers -> order_items ---
print("sellers -> order_items  (items fulfilled per seller):")
display(q("""
    WITH items_per_seller AS (
        SELECT s.seller_id,
               COUNT(oi.seller_id) AS item_count
        FROM raw.sellers s
        LEFT JOIN raw.order_items oi ON s.seller_id = oi.seller_id
        GROUP BY s.seller_id
    )
    SELECT
        MIN(item_count) AS min_items_fulfilled,
        MAX(item_count) AS max_items_fulfilled,
        SUM(CASE WHEN item_count = 0 THEN 1 ELSE 0 END) AS sellers_with_zero_items
    FROM items_per_seller
"""))
-- Result: min=1  max=2033  zeros=0
-- => Every seller fulfilled at least one item  -> sellers ||--|{ order_items

---
## Exercise 5 – Customer Identifier Analysis

The `customers` table has **two columns that look like identifiers**: `customer_id` and `customer_unique_id`.  
Counting distinct values reveals how they differ and which one should be used to count customers.

In [ ]:
# Exercise 5 – customer_id vs customer_unique_id
#
# customer_id       : a per-ORDER anonymisation key. Each order generates a new customer_id,
#                     so this column is unique at the row level but does NOT identify a person.
# customer_unique_id: identifies the same real-world person across multiple orders.
#                     Use COUNT(DISTINCT customer_unique_id) to count real customers.

print("Distinct counts – customer_id vs customer_unique_id")
display(q("""
    SELECT
        COUNT(*)                                                         AS total_rows,
        COUNT(DISTINCT customer_id)                                      AS distinct_customer_id,
        COUNT(DISTINCT customer_unique_id)                               AS distinct_customer_unique_id,
        COUNT(DISTINCT customer_id) - COUNT(DISTINCT customer_unique_id) AS difference
    FROM raw.customers
"""))
-- Result: total=99441  customer_id=99441  customer_unique_id=96096  difference=3345
-- => 3345 unique people placed more than one order during this period

print("Top 10 repeat buyers (customer_unique_id with multiple orders):")
display(q("""
    SELECT
        customer_unique_id,
        COUNT(DISTINCT customer_id) AS orders_placed
    FROM raw.customers
    GROUP BY customer_unique_id
    HAVING COUNT(DISTINCT customer_id) > 1
    ORDER BY orders_placed DESC
    LIMIT 10
"""))